## Install Dependencies


In [1]:
# Dependencis for the notebook
%pip install spacy pandas torch transformers huggingface_hub matplotlib ipympl rdflib sentence_transformers numpy
# optuna and optional dependencies
%pip install optuna scikit-learn plotly 
# Qwen optimization (optional) dependencies
%pip install flash-linear-attention causal-conv1d
# Mistral dependencies
%pip install accelerate


/home/rpatroni/bzk-post-processing/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/home/rpatroni/bzk-post-processing/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/home/rpatroni/bzk-post-processing/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/home/rpatroni/bzk-post-processing/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import matplotlib.pyplot as plt
import matplotlib
import optuna.visualization
import optuna.importance
import json
from pathlib import Path
from collections import OrderedDict
from modules.llms import (
    JSONTuplesPromptTemplate,
    JsonDictPromptTemplate,
    PromptTemplate,
    LLMAddressParsingModel,
    QwenAddressParsingModel,
    ZeroShot,
    SimilarExamples,
    FixedExamples,
    NERPatternSimilarExamples,
    FallbackExamplesStrategy,
    HybridSimilarExamples,
    FIXED_DEMO_EXAMPLES
)
from modules.deepparse_parser import DeepParseParser
from modules.libpostal_client import LibpostalClient
from modules.utils import compare_preds, format_time, natural_casing
from typing import NamedTuple
from tqdm.auto import tqdm

import abc
import pandas as pd
import time
import pprint
import textwrap
import numpy as np
import modules.train_ner as train_ner

# Total number of addresses in the complete BZK data for calculating estimated time of processing
TOTAL_ADDRESSES = 4_394_539

/home/rpatroni/bzk-post-processing/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


## Set hardware accelerator


In [3]:
# List available hardware accelerators for PyTorch
import torch
available_accelerators = []
if torch.cuda.is_available():
    print("CUDA - available devices:")
    for i in range(torch.cuda.device_count()):
        print(f"  {i}: {torch.cuda.get_device_name(i)}")
        available_accelerators.append(torch.device(f'cuda:{i}'))
elif torch.accelerator.is_available(): # Support other hardware accelators
    available_accelerators.append(torch.accelerator.current_accelerator())
else:
    print("WARNING: Running on CPU")
    device = torch.device('cpu')

CUDA - available devices:
  0: NVIDIA A100 80GB PCIe


/home/rpatroni/bzk-post-processing/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


In [4]:
if available_accelerators:
    device = available_accelerators[0]
print(f"Torch version: {torch.__version__}, Device: {device}")

Torch version: 2.10.0+cu128, Device: cuda:0


## Load Dataset

Read all splits and concat them

In [5]:
csv_read_args = dict(keep_default_na=False, dtype=str, na_values=[""])

bzkopen_train = pd.read_csv("open_data/bzkopen_addresses_train.csv", **csv_read_args)
bzkopen_val = pd.read_csv("open_data/bzkopen_addresses_val.csv", **csv_read_args)
bzkopen_test = pd.read_csv("open_data/bzkopen_addresses_val.csv", **csv_read_args)
bzkopen_merged : pd.DataFrame = pd.concat(
    [bzkopen_train, bzkopen_val, bzkopen_test], 
    keys=['train', 'val', 'test'], 
    names=['split', 'row']
)
display(bzkopen_merged.sample(5))

field                           FullAddress  \
split row                                                                  
train 658  ApplicantCurrentAddress    2434 e Ave. Cedar Rapids, Iowa/USA   
      355      ApplicantBirthPlace                      Rostoko/Bukowina   
      36       ApplicantBirthPlace                        Aschbach/Pfalz   
      113  ApplicantCurrentAddress  Eitorf, Bahnhofstr. 35 Reg.Bez. Köln   
      25          VictimDeathPlace                  Auschwitz umgekommen   

          UnitNumber HouseNumber   StreetName Neighborhood          City  \
split row                                                                  
train 658        NaN        2434       e Ave.          NaN  Cedar Rapids   
      355        NaN         NaN          NaN          NaN       Rostoko   
      36         NaN         NaN          NaN          NaN      Aschbach   
      113        NaN          35  Bahnhofstr.          NaN        Eitorf   
      25         NaN         NaN          NaN          NaN           NaN   

          District    Region State Country PostalCode  
split row                                              
train 658      NaN       NaN  Iowa     USA        NaN  
      355      NaN  Bukowina   NaN     NaN        NaN  
      36       NaN     Pfalz   NaN     NaN        NaN  
      113     Köln       NaN   NaN     NaN        NaN  
      25       NaN       NaN   NaN     NaN        NaN

## Create folds



In [6]:
N_FOLDS = 10

# Set shuffle seed for reproducibility
SEED=4841762


In [7]:
# Define paths for saving models and predictions
models_path = Path("models") / "cross_val" / f"{N_FOLDS}_folds" / f"seed_{SEED}"
models_path.mkdir(parents=True, exist_ok=True)
preds_path = Path("experiments_data") / "cross_val" / f"{N_FOLDS}_folds" / f"seed_{SEED}"
preds_path.mkdir(parents=True, exist_ok=True)

In [8]:
class Fold(NamedTuple):
    fold_idx: int
    train_data: pd.DataFrame
    val_data: pd.DataFrame
    models_path: Path
    preds_path: Path

In [9]:

shuffled_bzkopen = bzkopen_merged.sample(frac=1, random_state=SEED)

fold_indices = np.array_split(shuffled_bzkopen.index, N_FOLDS)

folds = []

for fold_idx, val_indices in enumerate(fold_indices):
    train_indices = shuffled_bzkopen.index.difference(val_indices)
    fold_models_path = models_path / f"fold_{fold_idx}"
    fold_models_path.mkdir(parents=True, exist_ok=True)
    fold_preds_path = preds_path / f"fold_{fold_idx}"
    fold_preds_path.mkdir(parents=True, exist_ok=True)
    folds.append(Fold(
        fold_idx, 
        shuffled_bzkopen.loc[train_indices], 
        shuffled_bzkopen.loc[val_indices],
        fold_models_path, 
        fold_preds_path
    ))
display(pd.DataFrame([[len(x) for x in [fold.val_data, fold.train_data]] for fold in folds], columns=['val_fold_size', 'train_fold_size']))

,val_fold_size,train_fold_size
0,108,967
1,108,967
2,108,967
3,108,967
4,108,967
5,107,968
6,107,968
7,107,968
8,107,968
9,107,968


# Train Spacy NER models

These models are used for the pattern example matching strategy so they will be reused over multiple different experiments. Therefore it is useful to train them in advance.

In [10]:
for fold in tqdm(folds, unit="fold"):
    train_ner.train(
        n_iter=30,
        output_dir=fold.models_path / 'ner_bzk',
        train_df=fold.train_data,
        val_df=fold.val_data
    )

  0%|          | 0/10 [00:00<?, ?fold/s]

Loading base model: de_core_news_sm
Loading training data...
  Using provided training DataFrame with 967 rows
  934 examples (provided train) (33 skipped, 10 entity values not aligned)
Loading validation data...
  Using provided validation DataFrame with 108 rows
  105 examples (provided val) (3 skipped, 0 entity values not aligned)

Training for 30 iterations (dropout=0.3)...
  Freezing pipes: ['tok2vec', 'tagger', 'morphologizer', 'parser', 'lemmatizer', 'attribute_ruler']


  0%|          | 0/30 [00:00<?, ?it/s]

/home/rpatroni/bzk-post-processing/.venv/lib/python3.12/site-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Berlin-Zehlendorf, Fischerhüttenstr. 57" with entities "[(0, 6, 'City'), (7, 17, 'Neighborhood'), (19, 36,...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
/home/rpatroni/bzk-post-processing/.venv/lib/python3.12/site-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Wehr/Baden, Enkendorfstr. 15." with entities "[(0, 4, 'City'), (12, 25, 'StreetName'), (26, 28, ...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
/home/rpatroni/bzk-post-processing/.venv/lib/python3.12/site-packages/spacy/training/iob_utils.py:

  iter   1  loss= 1942.61  P=0.6181  R=0.6738  F1=0.6448 *
  iter   2  loss= 1151.13  P=0.7604  R=0.7082  F1=0.7333 *
  iter   3  loss=  866.44  P=0.7904  R=0.7768  F1=0.7835 *
  iter   4  loss=  719.33  P=0.7839  R=0.7940  F1=0.7889 *
  iter   5  loss=  567.45  P=0.7815  R=0.7983  F1=0.7898 *
  iter   6  loss=  516.53  P=0.7890  R=0.8026  F1=0.7957 *
  iter   7  loss=  416.14  P=0.8077  R=0.8112  F1=0.8094 *
  iter   8  loss=  368.14  P=0.8109  R=0.8283  F1=0.8195 *
  iter   9  loss=  328.18  P=0.8216  R=0.8498  F1=0.8354 *
  iter  10  loss=  284.58  P=0.8051  R=0.8155  F1=0.8102
  iter  11  loss=  269.92  P=0.8276  R=0.8240  F1=0.8258
  iter  12  loss=  191.49  P=0.8085  R=0.8155  F1=0.8120
  iter  13  loss=  175.66  P=0.8197  R=0.8197  F1=0.8197
  iter  14  loss=  184.30  P=0.8151  R=0.8326  F1=0.8238
  iter  15  loss=  166.99  P=0.8125  R=0.8369  F1=0.8245
  iter  16  loss=  126.78  P=0.8133  R=0.8412  F1=0.8270
  iter  17  loss=  149.90  P=0.8426  R=0.8498  F1=0.8462 *
  iter  18 

...

/home/rpatroni/bzk-post-processing/.venv/lib/python3.12/site-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Goslar. Jakobistr.36" with entities "[(0, 7, 'City'), (8, 18, 'StreetName'), (18, 20, '...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
/home/rpatroni/bzk-post-processing/.venv/lib/python3.12/site-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Bremen-St. Magnus, Zum Fichtenhof 25" with entities "[(0, 6, 'City'), (7, 17, 'Neighborhood'), (19, 33,...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
/home/rpatroni/bzk-post-processing/.venv/lib/python3.12/site-packages/spacy/training/iob_utils.py:149: UserWar

  iter   1  loss= 1942.11  P=0.6769  R=0.7154  F1=0.6957 *
  iter   2  loss= 1147.77  P=0.8082  R=0.8049  F1=0.8065 *
  iter   3  loss=  820.11  P=0.8016  R=0.8211  F1=0.8112 *
  iter   4  loss=  765.05  P=0.8144  R=0.8740  F1=0.8431 *
  iter   5  loss=  590.23  P=0.8161  R=0.8659  F1=0.8402
  iter   6  loss=  497.56  P=0.8485  R=0.9106  F1=0.8784 *
  iter   7  loss=  432.24  P=0.8378  R=0.8821  F1=0.8594
  iter   8  loss=  409.52  P=0.8327  R=0.8699  F1=0.8509
  iter   9  loss=  381.43  P=0.8643  R=0.9065  F1=0.8849 *
  iter  10  loss=  331.32  P=0.8606  R=0.8780  F1=0.8692
  iter  11  loss=  260.54  P=0.8488  R=0.8902  F1=0.8690
  iter  12  loss=  225.83  P=0.8649  R=0.9106  F1=0.8871 *
  iter  13  loss=  222.24  P=0.8467  R=0.8984  F1=0.8718
  iter  14  loss=  193.38  P=0.8716  R=0.9106  F1=0.8907 *
  iter  15  loss=  200.48  P=0.8550  R=0.9106  F1=0.8819
  iter  16  loss=  183.42  P=0.8889  R=0.9106  F1=0.8996 *
  iter  17  loss=  165.92  P=0.8795  R=0.8902  F1=0.8848
  iter  18  l

  0%|          | 0/30 [00:00<?, ?it/s]

/home/rpatroni/bzk-post-processing/.venv/lib/python3.12/site-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Frankfurt/Main, Sellerstr.30/Frankfurt/Main (Hesse..." with entities "[(0, 14, 'City'), (16, 26, 'StreetName'), (26, 28,...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
/home/rpatroni/bzk-post-processing/.venv/lib/python3.12/site-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Tel Aviv/Israel. Javnestr: 16." with entities "[(0, 8, 'City'), (9, 15, 'Country'), (17, 25, 'Str...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
/home/rpatroni/bzk-post-processing/.venv/lib/python3.12/site-packages/spacy/trainin

  iter   1  loss= 1952.08  P=0.7459  R=0.6389  F1=0.6883 *
  iter   2  loss= 1154.19  P=0.8054  R=0.8241  F1=0.8146 *
  iter   3  loss=  900.21  P=0.8394  R=0.8472  F1=0.8433 *
  iter   4  loss=  721.05  P=0.8786  R=0.8380  F1=0.8578 *
  iter   5  loss=  606.93  P=0.8500  R=0.8657  F1=0.8578
  iter   6  loss=  540.89  P=0.8475  R=0.8750  F1=0.8610 *
  iter   7  loss=  450.95  P=0.8894  R=0.8935  F1=0.8915 *
  iter   8  loss=  410.37  P=0.8935  R=0.8935  F1=0.8935 *
  iter   9  loss=  397.97  P=0.8818  R=0.8981  F1=0.8899
  iter  10  loss=  336.43  P=0.8682  R=0.8843  F1=0.8761
  iter  11  loss=  334.87  P=0.8818  R=0.8981  F1=0.8899
  iter  12  loss=  263.01  P=0.8802  R=0.8843  F1=0.8822
  iter  13  loss=  192.77  P=0.8750  R=0.8750  F1=0.8750
  iter  14  loss=  202.46  P=0.8879  R=0.8796  F1=0.8837
  iter  15  loss=  169.32  P=0.8824  R=0.9028  F1=0.8924
  iter  16  loss=  168.99  P=0.8778  R=0.8981  F1=0.8879
  iter  17  loss=  153.24  P=0.8705  R=0.9028  F1=0.8864
  iter  18  loss=